# 04 Compare DRLB States vs Linear Baseline

Loads May 04 DRLB run summaries, evaluates linear baseline, and compares val/holdout metrics.

May 04 profiles (`may04_*`) set `lambda_min=float('-inf')` and `lambda_max=float('+inf')` so DRLB does not apply finite λ bounds.


In [ ]:
import sys
import json
import pickle
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
repo_root = cwd
while repo_root != repo_root.parent and not (repo_root / 'pyproject.toml').exists():
    repo_root = repo_root.parent
if not (repo_root / 'pyproject.toml').exists():
    raise RuntimeError('Could not locate repository root with pyproject.toml')

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from example_notebooks.experiments.adapters.baseline_adapter import evaluate_baseline_model_inprocess
from example_notebooks.experiments.drlb.profiles import build_config as build_drlb_config
from example_notebooks.experiments.infra.split_utils import resolve_normalized_splits


In [ ]:
drlb_runs = {
    'default': 'may04_default_state_optuna10',
    'ratio_bat': 'may04_ratio_bat_state_optuna10',
    'ta_ratio_bat': 'may04_ta_ratio_bat_state_optuna10',
}

config_ref = build_drlb_config(
    run_name='may04_compare_states_vs_linear',
    profile='may04_default_linear_lambda_legacy',
    split_set='full_train_val_holdout',
)
normalized_splits = resolve_normalized_splits(config_ref)

summary_rows = []
for state_key, run_name in drlb_runs.items():
    summary_path = config_ref.family_dir / run_name / 'outputs' / 'run_summary.json'
    payload = json.loads(summary_path.read_text())
    summary_rows.append({
        'model': f'drlb_{state_key}',
        'run_name': run_name,
        'state': state_key,
        'val_clicks_sum': payload['tuning']['best_val_metrics']['clicks_sum'],
        'val_cpc_relative': payload['tuning']['best_val_metrics']['cpc_relative'],
        'val_rmse': payload['tuning']['best_val_metrics']['rmse'],
        'val_quickspend': payload['tuning']['best_val_metrics']['quickspend'],
        'holdout_clicks_sum': payload['final_holdout']['metrics']['clicks_sum'],
        'holdout_cpc_relative': payload['final_holdout']['metrics']['cpc_relative'],
        'holdout_rmse': payload['final_holdout']['metrics']['rmse'],
        'holdout_quickspend': payload['final_holdout']['metrics']['quickspend'],
        'diagnostics_png': payload['refit']['combined_diagnostics_plot_path'],
    })

drlb_df = pd.DataFrame(summary_rows)
drlb_df


In [ ]:
linear_params_path = repo_root / 'example_notebooks' / 'evaluate_baselines' / 'best_params' / 'fpa_baseline_n10_rndm_42' / 'linear_scr_FPA.pkl'
with linear_params_path.open('rb') as f:
    linear_params = pickle.load(f)

linear_val = evaluate_baseline_model_inprocess(
    model_name='linear',
    label='linear_val',
    params_dict=linear_params,
    split=normalized_splits['val'],
    auction_mode='FPA',
)
linear_holdout = evaluate_baseline_model_inprocess(
    model_name='linear',
    label='linear_holdout',
    params_dict=linear_params,
    split=normalized_splits['test_holdout'],
    auction_mode='FPA',
)

linear_row = {
    'model': 'linear',
    'run_name': 'fpa_baseline_n10_rndm_42',
    'state': 'baseline',
    'val_clicks_sum': linear_val['metrics']['clicks_sum'],
    'val_cpc_relative': linear_val['metrics']['cpc_relative'],
    'val_rmse': linear_val['metrics']['rmse'],
    'val_quickspend': linear_val['metrics']['quickspend'],
    'holdout_clicks_sum': linear_holdout['metrics']['clicks_sum'],
    'holdout_cpc_relative': linear_holdout['metrics']['cpc_relative'],
    'holdout_rmse': linear_holdout['metrics']['rmse'],
    'holdout_quickspend': linear_holdout['metrics']['quickspend'],
    'diagnostics_png': None,
}

comparison_df = pd.concat([pd.DataFrame([linear_row]), drlb_df], ignore_index=True)
comparison_df


In [ ]:
linear_ref = comparison_df[comparison_df['model'] == 'linear'].iloc[0]

delta_rows = []
for _, row in comparison_df[comparison_df['model'] != 'linear'].iterrows():
    delta_rows.append({
        'model': row['model'],
        'val_clicks_delta_vs_linear': row['val_clicks_sum'] - linear_ref['val_clicks_sum'],
        'val_cpc_relative_delta_vs_linear': row['val_cpc_relative'] - linear_ref['val_cpc_relative'],
        'val_rmse_delta_vs_linear': row['val_rmse'] - linear_ref['val_rmse'],
        'val_quickspend_delta_vs_linear': row['val_quickspend'] - linear_ref['val_quickspend'],
        'holdout_clicks_delta_vs_linear': row['holdout_clicks_sum'] - linear_ref['holdout_clicks_sum'],
        'holdout_cpc_relative_delta_vs_linear': row['holdout_cpc_relative'] - linear_ref['holdout_cpc_relative'],
        'holdout_rmse_delta_vs_linear': row['holdout_rmse'] - linear_ref['holdout_rmse'],
        'holdout_quickspend_delta_vs_linear': row['holdout_quickspend'] - linear_ref['holdout_quickspend'],
    })

delta_df = pd.DataFrame(delta_rows).sort_values('holdout_clicks_delta_vs_linear', ascending=False)
best_state_row = comparison_df[comparison_df['model'] != 'linear'].sort_values('holdout_clicks_sum', ascending=False).iloc[0]

print('Best DRLB by holdout clicks_sum:', best_state_row['model'])
print('Best holdout clicks_sum:', best_state_row['holdout_clicks_sum'])

delta_df
